In [5]:
#!pip install wikipedia-api neo4j

In [5]:
#!pip install openai

In [6]:
import wikipediaapi
from neo4j import GraphDatabase
from openai import OpenAI
import re
import os
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### LLM NER Function

In [9]:
def extract_entities(text):
    prompt = f"""
    Extract entities and their types from the following text: \n\n{text}\n\n
    Format: Entity - Type (e.g. John Doe - Person)
    """
    response = client.chat.completions.create(
        model = "gpt-5-nano",
        messages = [
            {"role": "user", "content": prompt}
        ],
    )
    return response.choices[0].message.content.strip()

In [10]:
test_text = "John Doe is a software engineer at Google and has a degree in Computer Science from MIT."
print(extract_entities(test_text))

John Doe - Person
Google - Organization
MIT - Organization
Computer Science - Field
software engineer - Occupation


### NEO4J Connection

In [26]:
# Copied from Neo4j Sandbox
uri = "bolt://44.202.44.181:7687"
username = "neo4j"
password = "retractor-hand-washing"
driver = GraphDatabase.driver(uri, auth=(username, password))

In [19]:
def create_node(tx, label, name):
    query = f"MERGE (n:{label} {{name: $name}})"
    tx.run(query, name=name)

In [21]:
def create_relationship(tx, entity1, entity2, relationship):
    query = f"""
    MATCH (a {{name: $entity1}}), (b {{name: $entity2}})
    MERGE (a) - [r:{relationship}] -> (b)
    RETURN type(r)
    """
    tx.run(query, entity1=entity1, entity2=entity2)

### FETCH WIKIPEDIA Data

In [14]:
wiki = wikipediaapi.Wikipedia(language='en', user_agent="my_bot (abc@example.com)")
page_title = "Artificial Intelligence"
page = wiki.page(page_title)
if page.exists():
    wiki_text = page.text
    print(f"Text from {page_title}: {wiki_text[:100]}")
else:
    print(f"{page_title} does not exist")

Text from Artificial Intelligence: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically a


In [16]:
entities_text = extract_entities(wiki_text[:1000]) # capped for demo
print(f"Entities extracted from text: \n{entities_text}")

Entities extracted from text: 
Artificial intelligence - Concept
AI - Acronym
Google Search - Product
YouTube - Organization
Amazon - Organization
Netflix - Organization
Google Assistant - Product
Siri - Product
Alexa - Product
Waymo - Organization
language models - Technology
AI art - Concept
chess - Game
Go - Game
autonomous vehicles - Concept


### PARSE and CLEAN

In [17]:
def clean_entity_name(entity_name):
    return re.sub(r"^\d+\.\s*","",entity_name).strip()

def parse_entities(entities_text):
    entity_pattern = r"(.+) - (.+)"
    entities = []
    for line in entities_text.split("\n"):
        match = re.match(entity_pattern, line)
        if match:
            entity_name, entity_type = match.groups()
            entity_name = clean_entity_name(entity_name)
            entities.append((entity_name, entity_type.strip()))
    return entities

entities = parse_entities(entities_text)
entities

[('Artificial intelligence', 'Concept'),
 ('AI', 'Acronym'),
 ('Google Search', 'Product'),
 ('YouTube', 'Organization'),
 ('Amazon', 'Organization'),
 ('Netflix', 'Organization'),
 ('Google Assistant', 'Product'),
 ('Siri', 'Product'),
 ('Alexa', 'Product'),
 ('Waymo', 'Organization'),
 ('language models', 'Technology'),
 ('AI art', 'Concept'),
 ('chess', 'Game'),
 ('Go', 'Game'),
 ('autonomous vehicles', 'Concept')]

### Create Nodes and Relationships in NEO4J

In [22]:
with driver.session() as session:
    for i, (entity_name, entity_type) in enumerate(entities):
        session.execute_write(create_node, entity_type, entity_name)

        for j, (entity_name2, entity_type2) in enumerate(entities):
            if i!=j:
                if entity_type == "Application" and entity_type2 == "Concept":
                    session.execute_write(create_relationship, entity_name, entity_name2, "APPLIES_TO")
                if entity_type == "Acronym" and entity_type2 == "Concept":
                    session.execute_write(create_relationship, entity_name, entity_name2, "ABBREVIATION_OF")
                if entity_type == "Technology" and entity_type2 == "Concept":
                    session.execute_write(create_relationship, entity_name, entity_name2, "ENABLES")
                if entity_type == "Concept" and entity_type2 == "Game":
                    session.execute_write(create_relationship, entity_name, entity_name2, "APPLIED_IN")
                if entity_type == "Organization" and entity_type2 == "Concept":
                    session.execute_write(create_relationship, entity_name, entity_name2, "USES")
                if entity_type == "Product" and entity_type2 == "Technology":
                    session.execute_write(create_relationship, entity_name, entity_name2, "POWERED_BY")

In [28]:
# query the graph
read_query = """
MATCH (ac: Acronym) - [:ABBREVIATION_OF] -> (c: Concept)
RETURN c.name as concept, ac.name as acronym
"""
with driver.session() as session:
    data = session.run(read_query)
    for record in data:
        print(record)

<Record concept='Artificial intelligence' acronym='AI'>
